### README

ML with XGBoost

1. Preprocess data and split into training and testing sets
2. Train base model
3. Perform hyperparameter tuning with Optuna
4. Train final model
5. Assess final model predictions on the test dataset
6. Interpret feature importance on the trained dataset using SHAP
7. Save the final model

NOTE: Replace /PATH/TO/... with the actual paths on your system.


### Requirements

In [ ]:
# Install

"""
joblib==1.5.1
matplotlib==3.10.0
numpy==2.0.2
optuna==4.2.1
pandas==2.2.3
scikit-learn==1.6.1
scipy==1.15.2
seaborn==0.13.2
shap==0.46.0
xgboost==2.1.4

"""

In [ ]:
# Import libraries

import numpy as np
import pandas as pd

import xgboost as xgb
import optuna
from sklearn.model_selection import cross_val_score, train_test_split, RepeatedStratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, make_scorer, matthews_corrcoef,
    classification_report, confusion_matrix, precision_recall_curve)

import shap
from scipy.stats import zscore

import seaborn as sns 
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib as mpl

import joblib

### Constants

In [ ]:
# Input: selected data from "data selection" step
DATA_PATH = '/PATH/TO/ml_input.csv'

# Output: set directory
OUTPUT_PATH = '/PATH/TO/OUTPUT'

# Set colours for each base (for plots)
BASECOLOURS_DICT = {'C': 'royalblue', 'U': 'firebrick', 'G': 'xkcd:tangerine', 'A': 'forestgreen'}

# Set col with group classification
CLASSIFICATION_COL = 'Regulation'  

# List with values to be treated as categories
CATEGORY_GROUPS = [('A', 'C', 'G', 'U'), (0, 1)]                                 

# Set objective type of xgboost 
OBJECTIVE_XGBOOST = 'binary:logistic'

# Set a float to define test proportion split
TEST_SIZE = 0.3

# Define cross-validation strategy
CROSS_VALIDATION = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

# Set group of features to be used
FEATURES = '5_LC' # choose between: 'all', '3_BP' (3' splice site and branch point related features) or '5_LC' (5' splice site and local context related features)

# Set metrics to guide model training and hyperparameter optimization
EVAL_METRIC_XGBOOST = 'logloss' 
EVAL_METRIC_OPTUNA = make_scorer(matthews_corrcoef) 
DIRECTION_OPTUNA = 'maximize' # 'maximize' 'minimize' for optuna eval metric


### Functions

##### Model

In [ ]:
#--------------------------------------------------------------------------------------------------
# Select features to be used in model training based on mode
#--------------------------------------------------------------------------------------------------

def select_features(df, classification_col, mode='all'):
    
    if classification_col not in df.columns:
        raise ValueError(f"Column '{classification_col}' not found!")

    # set list of all features
    all_cols = df.columns.tolist()
    features = [col for col in all_cols if col != classification_col]

    # select final features based on mode
    if mode == 'all':
        selected_cols = features
    elif mode == '3_BP':
        selected_cols = [col for col in features if '5end' not in col and 'GC' not in col and 'Len' not in col] 
    elif mode == '5_LC':
        selected_cols = [col for col in features if '5end' in col or 'GC' in col or 'Len' in col] 
    else:
        raise ValueError(f"Invalid mode type: '{mode}'.")

    return df[selected_cols].copy()


#--------------------------------------------------------------------------------------------------
# Ensure correct column data types for XGBoost with categorical features
#--------------------------------------------------------------------------------------------------

def preprocess_cols_types(df: pd.DataFrame, category_groups: list[tuple | list]):
    
    df = df.copy()

    # Pre-build categorical dtypes
    cat_dtypes = {
        tuple(group): pd.CategoricalDtype(categories=list(group))
        for group in category_groups
    }

    for col in df.columns:
        values = df[col].dropna().unique()

        # Skip empty columns
        if len(values) == 0:
            continue

        matched = False

        for group, cat_dtype in cat_dtypes.items():
            # Normalise string categories
            if all(isinstance(v, str) for v in values):
                values_norm = pd.Series(values).astype(str).str.upper().unique()
                group_norm = [str(g).upper() for g in group]
            else:
                values_norm = values
                group_norm = group

            # Check if column values fit entirely in the category group
            if set(values_norm).issubset(set(group_norm)):
                df[col] = (
                    df[col].astype(str).str.upper() # uppercase if all values in that col are strings
                    if all(isinstance(v, str) for v in values)
                    else df[col]
                )
                df[col] = df[col].astype(cat_dtype)
                matched = True
                break

        # Treat col as numerical float if values not in category groups
        if not matched:
            if pd.api.types.is_numeric_dtype(df[col]):
                df[col] = df[col].astype(float)

    return df


#--------------------------------------------------------------------------------------------------
# Evaluate a model using cross-validation with multiple scoring metrics
#--------------------------------------------------------------------------------------------------

def evaluate_model_cv(model, X, y, cv, model_name="Model"):

    print(f"Evaluating {model_name} with cross-validation...\n")

    output = []

    scorers = {
        'MCC': make_scorer(matthews_corrcoef, greater_is_better=True),
        'Accuracy': make_scorer(accuracy_score),
        'Precision': make_scorer(precision_score),
        'Recall': make_scorer(recall_score),
        'F1': make_scorer(f1_score)
    }


    for name, scorer in scorers.items():
        scores = cross_val_score(model, X, y, cv=cv, scoring=scorer, n_jobs=-1)
        line = f"{name:10}: {np.mean(scores):.3f} ± {np.std(scores):.3f}"
        print(line)
        output.append(line)

    # Log loss is handled separately because it's a loss (lower is better)
    log_loss_scores = cross_val_score(model, X, y, cv=cv, scoring='neg_log_loss', n_jobs=-1)
    log_loss_scores = -log_loss_scores  # Flip sign to make it positive
    line = f"{'LogLoss':10}: {np.mean(log_loss_scores):.3f} ± {np.std(log_loss_scores):.3f}"
    print(line)
    output.append(line)
    return output

#--------------------------------------------------------------------------------------------------
# Create an Optuna objective function for hyperparameter tuning
#--------------------------------------------------------------------------------------------------

def make_objective(model_class, X, y, cv, eval_metric_xgboost, eval_metric_optuna):
    
    def objective(trial):
        params = {
            'verbosity': 0,
            'eval_metric': eval_metric_xgboost,
            'objective': OBJECTIVE_XGBOOST,
            'enable_categorical': True,
            'random_state': 42,
            'n_estimators': trial.suggest_int('n_estimators', 50, 200),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'gamma': trial.suggest_float('gamma', 0, 5),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            #'max_delta_step': trial.suggest_int('max_delta_step', 1)
        }
        

        model = model_class(**params)
        score = cross_val_score(model, X, y, cv=cv, scoring=eval_metric_optuna, n_jobs=-1, verbose=1)
        return score.mean()

    return objective


##### Plot

In [ ]:
#--------------------------------------------------------------------------------------------------
# Plot SHAP scatter 
#--------------------------------------------------------------------------------------------------

def plot_shap_scatter(
    model,
    input_df,
    feature_type='base',        # 'base' or 'numeric'
    basecolours_dict=None,      # required if feature_type='base'
    topfeat_n=6,
    figsize=(5, 5),
    cmap='coolwarm',
    title=None,
    save_path=None
):
    
    
    # Calculate SHAP
    explainer = shap.TreeExplainer(model)
    shap_values = explainer(input_df).values  
    
    # Determine features to plot
    if feature_type == 'base':
        if basecolours_dict is None:
            raise ValueError("For feature type 'base', provide a dictionary with colours")
        plot_cols = [col for col in input_df.columns if 'base' in col]
        sort_key = lambda x: x.cat.codes
    elif feature_type == 'numeric':
        plot_cols = input_df.select_dtypes(include='number').columns
        sort_key = None
    else:
        raise ValueError("feature_type deve ser 'base' ou 'numeric'")
    
    # Create df with features to plot
    plot_data = []
    for feature in plot_cols:
        temp_df = pd.DataFrame({
            'Feature': feature,
            'SHAP_Value': shap_values[:, input_df.columns.get_loc(feature)],
            'Category': input_df[feature]
        })
        plot_data.append(temp_df)
    
    plot_df = pd.concat(plot_data)
    
    # Plot top n features
    shap_sum = plot_df.groupby('Feature')['SHAP_Value'].apply(lambda x: x.abs().sum())
    top_features = shap_sum.sort_values(ascending=False).head(topfeat_n).index
    plot_df = plot_df[plot_df['Feature'].isin(top_features)]
    # Sort feature order to plot
    plot_df['Feature'] = pd.Categorical(plot_df['Feature'], categories=top_features, ordered=True)
    plot_df = plot_df.sort_values(by='Feature')
    
    # Prepare subplots
    unique_features = plot_df['Feature'].unique()
    n_features = len(unique_features)
    rows = n_features + 1 # +1 for the final x axis
    fig, axes = plt.subplots(rows, 1, figsize=(figsize[0], figsize[1]*rows))
    axes = axes.flatten()
    abs_shapmax = np.ceil(np.max(np.abs(plot_df['SHAP_Value'])))
    
    # Plotting
    for i, column in enumerate(unique_features):
        df = plot_df[plot_df['Feature'] == column].copy()
        jitter = np.random.normal(0, 0.2, size=df.shape[0])
        
        if feature_type == 'base':
            df = df.sort_values(by='Category', key=sort_key)
            sns.scatterplot(
                data=df,
                x='SHAP_Value',
                y=np.linspace(0, 1, df.shape[0]) + jitter,
                ax=axes[i],
                s=30,
                hue='Category',
                palette=basecolours_dict,
                alpha=1,
                edgecolor='none'
            )
        else:  # numeric
            df = df.sort_values(by='Category', ascending=True)
            # check if numeric values are the same
            if np.std(df['Category']) > 1e-8: # not constant, use colour scale
                df['Category'] = zscore(df['Category'])
                sc = axes[i].scatter(
                    df['SHAP_Value'],
                    np.linspace(0, 1, df.shape[0]) + jitter,
                    c=df['Category'],
                    cmap=cmap,
                    s=30,
                    alpha=1
                )
            else: # all values are the same, use gray colour only
                df['Category'] = 1
                sc = axes[i].scatter(
                    df['SHAP_Value'],
                    np.linspace(0, 1, df.shape[0]) + jitter,
                    c="gray",
                    s=30,
                    alpha=1
                )
        
        # Axis settings
        axes[i].set_xlim(-abs_shapmax, abs_shapmax)
        axes[i].set_ylim(-1, 2)
        axes[i].set_xlabel('')
        axes[i].set_ylabel(column, rotation=0, labelpad=20, ha='right')
        axes[i].tick_params(axis='y', which='both', left=False, labelleft=False)
        axes[i].tick_params(axis='x', which='both', bottom=False, labelbottom=False)
        axes[i].spines[['left', 'top', 'right', 'bottom']].set_visible(False)
        axes[i].axvline(x=0, linestyle='-', c="black", linewidth=1, zorder=10, alpha=0.7)
        axes[i].axhline(y=0.5, color='grey', alpha=0.7, linewidth=1, linestyle=':', zorder=0)
        if axes[i].get_legend() and feature_type=='base':
            axes[i].legend_.set_visible(False)

    # Last axis with X label
    axes[-1].set_xlabel('SHAP Value')
    axes[-1].set_xlim(-abs_shapmax, abs_shapmax)
    axes[-1].spines[['left', 'top', 'right']].set_visible(False)
    axes[-1].tick_params(axis='y', which='both', left=False, labelleft=False)
    
    # Add legend or colour bar
    if feature_type == 'base':
        handles = [
            mlines.Line2D([], [], marker='o', color=basecolours_dict[b], label=b, linestyle='None', markersize=10)
            for b in basecolours_dict
        ]
        fig.legend(handles=handles, loc='upper center', ncol=4, bbox_to_anchor=(0.55, -0.01), frameon=False)
    else:  # numeric
        if input_df.shape[0] > 2:
            norm = mpl.colors.Normalize(vmin=-1, vmax=1)
            sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
            sm.set_array([])
            cbar_ax = fig.add_axes([0.1, -0.05, 0.8, 0.02])
            cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
            cbar.set_ticks([cbar.vmin, cbar.vmax])
            cbar.set_ticklabels(['Low', 'High'])
    
    # Add title
    if title:
        plt.suptitle(title, y=1.03)

    # Save figure
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    plt.show()


In [ ]:
#--------------------------------------------------------------------------------------------------
# Plot precision recall curve 
#--------------------------------------------------------------------------------------------------

def plot_precision_recall_curve(classifier, X_test, y_test, threshold=0.5, figsize=(8,6), cmap="coolwarm", save_path=None):

    # Predicted probabilities for the positive class
    y_scores = classifier.predict_proba(X_test)[:, 1]
    
    # Compute Precision-Recall curve
    precision, recall, thresholds = precision_recall_curve(y_test, y_scores)
    thresholds = np.insert(thresholds, 0, 0)  # align with precision/recall arrays
    
    # Line if model with no skill 
    no_skill = np.mean(y_test)
    
    # Index of the highlighted threshold
    threshold_idx = np.argmin(np.abs(thresholds - threshold))
    
    # Plot Precision-Recall curve
    plt.figure(figsize=figsize)
    scatter = plt.scatter(
        recall, precision,
        c=thresholds, cmap=cmap,
        edgecolors='none', alpha=0.75,
        vmin=0, vmax=1
    )
    plt.colorbar(scatter, label="Threshold")
    plt.plot([0, 1], [no_skill, no_skill], linestyle=':', color='black', label='No Skill')
    
    # Circle the threshold used
    threshold_idx = np.argmin(np.abs(thresholds - threshold))
    plt.scatter(
        recall[threshold_idx], precision[threshold_idx],
        s=150, facecolors='none', edgecolors='grey', linewidths=1,
        label=f'Threshold {threshold:.2f}'
    )
    
    # Adjustments
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.ylim(0.4, 1.03)
    plt.title("Precision-Recall Curve (Test Set)")
    plt.grid(True)
    plt.legend(loc='lower left')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path)
    
    plt.show()


### Analysis

##### Settings


In [ ]:
# shared params for all models stages
shared_params = {
        'eval_metric': EVAL_METRIC_XGBOOST,
        'objective': OBJECTIVE_XGBOOST,
        'enable_categorical': True,
        'random_state': 42
}


In [ ]:
# save to file
with open(f"{OUTPUT_PATH}/ml_results.txt", "w") as f:
    f.write("--------------------------------------------------------------------------------\n")
    f.write("SETTINGS\n")
    f.write("--------------------------------------------------------------------------------\n\n")
    f.write(f"XGBoost: {EVAL_METRIC_XGBOOST}\n")
    f.write(f"Optuna: {EVAL_METRIC_OPTUNA}\n")
    f.write(f"Features type: {FEATURES}\n")
    f.write("\n\n")

##### Data preprocessing

In [ ]:
# Selects only introns classified as targets or non-targets
data_df = pd.read_csv(DATA_PATH)

# Separate features and target variable
X = select_features(df=data_df, classification_col=CLASSIFICATION_COL, mode=FEATURES)
y = data_df[CLASSIFICATION_COL]

# Assures categorical type correctly
X = preprocess_cols_types(X, category_groups=CATEGORY_GROUPS)

# Split data into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=42
)


##### Base model

In [ ]:
# Define base model for further optimisation
xgb_base = xgb.XGBClassifier(
    **shared_params
)

# Evaluate model
results = evaluate_model_cv(xgb_base, X_train, y_train, cv=CROSS_VALIDATION, model_name="Base XGBoost")

# save to file
with open(f"{OUTPUT_PATH}/ml_results.txt", "a") as f:
   f.write("--------------------------------------------------------------------------------\n")
   f.write("BASE MODEL\n")
   f.write("--------------------------------------------------------------------------------\n\n")
   f.write("\n".join(results))
   f.write("\n\n\n")

##### Hyperparameter tuning

In [ ]:
# Prepare your variables: model_class, X_train_selected_features, y_train, cv (e.g. StratifiedKFold)
objective = make_objective(xgb.XGBClassifier, X_train, y_train, CROSS_VALIDATION, EVAL_METRIC_XGBOOST, EVAL_METRIC_OPTUNA)

study = optuna.create_study(direction=DIRECTION_OPTUNA, sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, n_jobs=1)

print("Best hyperparameters found:")
print(study.best_params)


# save to file
with open(f"{OUTPUT_PATH}/ml_results.txt", "a") as f:
    f.write("--------------------------------------------------------------------------------\n")
    f.write("HYPERPARAMETER TUNING\n")
    f.write("--------------------------------------------------------------------------------\n\n")
    f.write("Best parameters found:\n")
    f.write(str(study.best_params))
    f.write("\n\n\n")

##### Train final model

In [ ]:
# Train model
best_params = study.best_params
xgb_hyper = xgb.XGBClassifier(
    **best_params,
    **shared_params
)
classifier = xgb_hyper.fit(X_train, y_train)

# Evaluate model
results = evaluate_model_cv(xgb_hyper, X_train, y_train, cv=CROSS_VALIDATION, model_name="XGBoost after hyperparameter tuning")

# Save to file
with open(f"{OUTPUT_PATH}/ml_results.txt", "a") as f:
    f.write("--------------------------------------------------------------------------------\n")
    f.write("FINAL MODEL\n")
    f.write("--------------------------------------------------------------------------------\n\n")
    f.write("\n".join(results))
    f.write("\n\n\n")

##### Final model evaluation on the test dataset

In [ ]:
# Final model report
classifier_pred = classifier.predict(X_test)
print('Classification report:')
report = classification_report(y_test, classifier_pred)
print(report)

# Compute confusion matrix
cm = confusion_matrix(y_test, classifier_pred)
print("Confusion Matrix:")
print(cm)
mcc_score = matthews_corrcoef(y_test, classifier_pred)
print(f"Test MCC: {mcc_score:.4f}")


# save to file
with open(f"{OUTPUT_PATH}/ml_results.txt", "a") as f:
    f.write("--------------------------------------------------------------------------------\n")
    f.write("EVALUATION ON TEST DATASET\n")
    f.write("--------------------------------------------------------------------------------\n\n")
    f.write(str(report))
    f.write("\n\n")
    f.write("Confusion Matrix:\n")
    f.write(str(cm))
    f.write("\n\n")
    f.write(f"Test MCC: {mcc_score:.4f}")
    f.write("\n\n\n")



In [ ]:
# Precision recall curve plot with used threshold
plot_precision_recall_curve(
    classifier=classifier,
    X_test=X_test,
    y_test=y_test,
    threshold=0.5,
    save_path=f"{OUTPUT_PATH}/precision_recall_test.svg"
)


##### Feature importance

In [ ]:
explainer = shap.Explainer(classifier)
shap_values = explainer(X_train)

#------------------------------------------------------------------------------------------------

shap.plots.bar(shap_values, show=False, max_display=6)
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/SHAP_absmean.svg')
plt.show()

#------------------------------------------------------------------------------------------------

plot_shap_scatter(classifier, X_train, feature_type='base', basecolours_dict=BASECOLOURS_DICT, topfeat_n=3, figsize=(5, 1),
                            save_path=f'{OUTPUT_PATH}/SHAP_base.svg') 

#------------------------------------------------------------------------------------------------

plot_shap_scatter(classifier, X_train, feature_type='numeric', topfeat_n=3, figsize=(5, 1),
                            save_path=f'{OUTPUT_PATH}/SHAP_num.svg') 




##### Save final model

In [ ]:
joblib.dump(classifier, f"{OUTPUT_PATH}/xgb_model.pkl")
